In [ ]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# Computing the Conjunctive Normal Form in First Order Logic

In order to convert a formula $f$ from first order logic into a set of clauses that is satisfiable if and only if $f$ is satisfiable,
we have to perform the following steps in order:
- eliminate biconditionals, exclusive ors, and conditionals, i.e. we transform the formula into a *pure* formula,
- transform the formula into *negation normal form*,
  i.e. we push the negation symbol inwards,
- rename bound variables to avoid clashes, i.e. we transform the formula into a *rectified* formula,
- transform the formula into *prenex normal form*,
  i.e. we move the quantifiers outside,
- eliminate existential quantifiers by *skolemizing* the formula, and
- transform the formula into *clauses* in set notation.

For every intermediate form, we define a separate type.  The type of every function then states which kind of formula it expects and which kind of formula it returns.  Hence the F# compiler checks that the steps are performed in the right order.

Some of these steps only work if the formula is *rectified*, where we define a formula $f$ as *rectified* if all quantifiers appearing in $f$ bind **different** variables.  For example, the formula
$$ \bigl(\forall x: p(x)\bigr) \vee \bigl(\forall x: q(x)\bigr)$$
is **not** *rectified*, because there are two different universal quantifiers that both bind the same variable $x$.  We can rewrite this formula as a *rectified* formula by *renaming* all occurrences of $x$ that are bound by the second quantifier as follows:
$$ \bigl(\forall x: p(x)\bigr) \vee \bigl(\forall y: q(y)\bigr)$$
Even if the given formula is rectified, the elimination of biconditionals produces formulas that are not rectified, because this elimination duplicates subformulas.  Therefore, the bound variables are renamed after the formula has been transformed into negation normal form.

## Auxiliary Functions

Formulas are represented as values of the type `Formula` that is defined in the notebook `FOL-Parser.ipynb`, which also provides the function `parse` that converts a string into a formula.  The parser distinguishes variables and function symbols as follows:
- A word starting with a *lower* case letter is interpreted as a *variable*.
- A word starting with an *upper* case letter is assumed to be a *function* or *predicate symbol*.

In [ ]:
importNotebook "FOL-Parser.ipynb"

A formula in conjunctive normal form is represented in *set notation*, i.e. as a set of *clauses*, where every clause is a set of *literals*.  A literal is either an atomic formula or a negated atomic formula.  We define the following types:
- `Pos (p, [t1; ...; tn])` represents the literal $p(t_1, \cdots, t_n)$,
- `Neg (p, [t1; ...; tn])` represents the literal $\neg p(t_1, \cdots, t_n)$,
- a `Clause` is a set of literals.

These types are also used in the notebook `03-Prover.ipynb`.

In [ ]:
type Literal =
    | Pos of string * Term list   // p(t1, ..., tn)
    | Neg of string * Term list   // ¬p(t1, ..., tn)

type Clause = Set<Literal>

The functions `literalToString`, `clauseToString`, and `clausesToString` convert literals, clauses, and sets of clauses into readable strings.  The function `termToString` is defined in the notebook `FOL-Parser.ipynb`.

In [ ]:
let literalToString (l: Literal) : string =
    match l with
    | Pos (p, ts) -> termToString (Fun (p, ts))
    | Neg (p, ts) -> "¬" + termToString (Fun (p, ts))

let clauseToString (C: Clause) : string =
    "{" + String.concat ", " [ for l in C -> literalToString l ] + "}"

let clausesToString (Clauses: Set<Clause>) : string =
    "{\n" + String.concat ",\n" [ for C in Clauses -> "    " + clauseToString C ] + "\n}"

For testing purposes, the following formula is used.  This formula specifies the notion of a *grandparent*.

In [ ]:
let s  = "∀g:∀c:(Grandparent(g, c) ↔ ∃p: (Parent(g, p) ∧ Parent(p, c)))"
let f1 = parse s
f1

As the values of type `Formula` are hard to read, we will mostly display formulas using the function `formulaToString` from the notebook `FOL-Parser.ipynb`.

In [ ]:
formulaToString f1

A *substitution* $\sigma$ is represented as a map of the form $\{x_1 \mapsto s_1, \cdots, x_n \mapsto s_n\}$ that assigns terms to variables.

The function `applyTerm` takes two arguments:
- `σ` is a substitution,
- `t` is a term.

It returns the term $t\sigma$ that results from replacing every occurrence of a variable $x_i$ in `t` by the corresponding term $s_i$.

In [ ]:
let rec applyTerm (σ: Map<string, Term>) (t: Term) : Term =
    match t with
    | Var x       -> match σ.TryFind x with
                     | Some s -> s
                     | None   -> t
    | Fun (f, ts) -> Fun (f, List.map (applyTerm σ) ts)

## Pure Formulas

A formula is *pure* if it contains only atomic formulas, the constants `⊤` and `⊥`, the operators `¬`, `∧`, and `∨`, and quantifiers.  The type `PureFormula` represents pure formulas.  Its constructors are the constructors of the type `Formula` except for `Imp`, `Iff`, and `Xor`.

The attribute `[<RequireQualifiedAccess>]` requires that the constructors of `PureFormula` are always qualified with the name of the type, e.g. we have to write `PureFormula.And (f, g)` instead of `And (f, g)`.  Without this attribute, the constructors of `PureFormula` would hide the constructors of `Formula` that have the same names.  Then `And (f, g)` would no longer be a `Formula`, neither in this notebook nor in the notebooks that import it.

In [ ]:
[<RequireQualifiedAccess>]
type PureFormula =
    | Verum                                 // ⊤
    | Falsum                                // ⊥
    | Atom   of string * Term list          // P(t1, ..., tn)
    | Not    of PureFormula                 // ¬
    | And    of PureFormula * PureFormula   // ∧
    | Or     of PureFormula * PureFormula   // ∨
    | Forall of string * PureFormula        // ∀x: f
    | Exists of string * PureFormula        // ∃x: f

The function `pureToFormula` takes one argument:
- `f` is a pure formula.

It returns `f` as a value of the type `Formula`.  As every pure formula is also a formula, this conversion is straightforward.  We need it in order to display pure formulas with the function `formulaToString` from the notebook `FOL-Parser.ipynb`.

In [ ]:
let rec pureToFormula (f: PureFormula) : Formula =
    match f with
    | PureFormula.Verum         -> Verum
    | PureFormula.Falsum        -> Falsum
    | PureFormula.Atom (p, ts)  -> Atom (p, ts)
    | PureFormula.Not g         -> Not (pureToFormula g)
    | PureFormula.And (g, h)    -> And (pureToFormula g, pureToFormula h)
    | PureFormula.Or  (g, h)    -> Or  (pureToFormula g, pureToFormula h)
    | PureFormula.Forall (x, g) -> Forall (x, pureToFormula g)
    | PureFormula.Exists (x, g) -> Exists (x, pureToFormula g)

## Eliminating Biconditionals, Exclusive Ors, and Conditionals

The function `purify` takes one argument:
- `f` is a formula from first order logic.

It returns a pure formula that is equivalent to `f`, i.e. the operators '→', '↔', and '⊕' are eliminated.  This is done by using the following equivalences:
$$ (f \rightarrow g) \;\Leftrightarrow\; (\neg f \vee g) $$
$$ (f \leftrightarrow g) \;\Leftrightarrow\; (f \rightarrow g) \wedge (g \rightarrow f) $$
$$ (f \oplus g) \;\Leftrightarrow\; (f \wedge \neg g) \vee (\neg f \wedge g) $$

**Implementation:** As in the function `purify` for propositional logic, if the operator of `f` is '→', '↔', or '⊕', then `f` is first rewritten into an equivalent value of type `Formula` using the corresponding equivalence.  Then `purify` is applied to the rewritten formula.  Although the rewritten formula is bigger than `f`, the recursion terminates: after at most two rewriting steps, `purify` is only called with subformulas of `f`.

The equivalences for '↔' and '⊕' duplicate the formulas $f$ and $g$.  Hence, if $f$ or $g$ contain quantifiers, the result of `purify` is not rectified.  This is taken care of by the function `rectify`, which is defined below.

In [ ]:
let rec purify (f: Formula) : PureFormula =
    match f with
    | Verum         -> PureFormula.Verum
    | Falsum        -> PureFormula.Falsum
    | Atom (p, ts)  -> PureFormula.Atom (p, ts)
    | Not g         -> PureFormula.Not (purify g)
    | And (g, h)    -> PureFormula.And (purify g, purify h)
    | Or  (g, h)    -> PureFormula.Or  (purify g, purify h)
    | Imp (g, h)    -> purify (Or (Not g, h))
    | Iff (g, h)    -> purify (And (Imp (g, h), Imp (h, g)))
    | Xor (g, h)    -> purify (Or (And (g, Not h), And (Not g, h)))
    | Forall (x, g) -> PureFormula.Forall (x, purify g)
    | Exists (x, g) -> PureFormula.Exists (x, purify g)

In [ ]:
formulaToString f1

In [ ]:
let f2 = purify f1
formulaToString (pureToFormula f2)

The formula `f2` is not rectified: the variable `p` is bound by two different quantifiers.

## Negation Normal Form

A formula is in *negation normal form* if it is pure and the negation operator `¬` is only applied to atomic formulas.  Hence, a formula in negation normal form is built from literals and the constants `⊤` and `⊥` using the operators `∧` and `∨` and quantifiers.  The type `NNF` represents these formulas:
- `NNF.Lit l` represents the literal `l`, i.e. either an atomic formula or a negated atomic formula.
- `NNF` has no constructor for negation.  Therefore, a value of type `NNF` can not contain any negation except for the negations of atomic formulas that are part of literals.

The attribute `[<RequireQualifiedAccess>]` is needed for the same reason as for the type `PureFormula`.

In [ ]:
[<RequireQualifiedAccess>]
type NNF =
    | Lit    of Literal          // P(t1, ..., tn) or ¬P(t1, ..., tn)
    | Verum                      // ⊤
    | Falsum                     // ⊥
    | And    of NNF * NNF        // ∧
    | Or     of NNF * NNF        // ∨
    | Forall of string * NNF     // ∀x: f
    | Exists of string * NNF     // ∃x: f

The function `literalToFormula` takes one argument:
- `l` is a literal.

It returns `l` as a value of the type `Formula`.  The function `nnfToFormula` takes a formula `f` in negation normal form and returns `f` as a value of the type `Formula`.  We need these functions in order to display formulas in negation normal form.

In [ ]:
let literalToFormula (l: Literal) : Formula =
    match l with
    | Pos (p, ts) -> Atom (p, ts)
    | Neg (p, ts) -> Not (Atom (p, ts))

let rec nnfToFormula (f: NNF) : Formula =
    match f with
    | NNF.Lit l         -> literalToFormula l
    | NNF.Verum         -> Verum
    | NNF.Falsum        -> Falsum
    | NNF.And (g, h)    -> And (nnfToFormula g, nnfToFormula h)
    | NNF.Or  (g, h)    -> Or  (nnfToFormula g, nnfToFormula h)
    | NNF.Forall (x, g) -> Forall (x, nnfToFormula g)
    | NNF.Exists (x, g) -> Exists (x, nnfToFormula g)

The function `nnf` takes one argument:
- `f` is a pure formula, i.e. it contains only the operators `¬`, `∧`, `∨`, `⊤`, `⊥`, and quantifiers.

It returns the <em style="color:blue;">negation normal form</em> of $f$, while `neg f` returns the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(p) = p$ for all atomic formulas $p$, $\texttt{nnf}(\top) = \top$, and $\texttt{nnf}(\bot) = \bot$,</li>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) =
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) =
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
    <li> $\texttt{nnf}(\forall x: F) = \forall x: \texttt{nnf}(\texttt{F})$.</li>
    <li> $\texttt{nnf}(\exists x: F) = \exists x: \texttt{nnf}(\texttt{F})$.</li>
</ol>

The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all atomic formulas $p$,</li>
    <li> $\texttt{neg}(\top) = \bot$ and $\texttt{neg}(\bot) = \top$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\forall x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \forall x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\exists x: \neg F\bigr) \\[0.1cm]
       = & \exists x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \exists x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\forall x: F \bigr) = \exists x: \texttt{neg}(F)$.</li>
      <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\exists x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \exists x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\forall x: \neg F\bigr) \\[0.1cm]
       = & \forall x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \forall x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\exists x: F \bigr) = \forall x: \texttt{neg}(F)$.</li>
</ol>

The functions `nnf` and `neg` are mutually recursive and are therefore defined together using `let rec ... and ...`.  An atomic formula $p(t_1, \cdots, t_n)$ is mapped to the literal `Pos (p, [t1; ...; tn])` by `nnf` and to the literal `Neg (p, [t1; ...; tn])` by `neg`.

In [ ]:
let rec nnf (f: PureFormula) : NNF =
    match f with
    | PureFormula.Verum         -> NNF.Verum
    | PureFormula.Falsum        -> NNF.Falsum
    | PureFormula.Atom (p, ts)  -> NNF.Lit (Pos (p, ts))
    | PureFormula.Not g         -> neg g
    | PureFormula.And (g, h)    -> NNF.And (nnf g, nnf h)
    | PureFormula.Or  (g, h)    -> NNF.Or  (nnf g, nnf h)
    | PureFormula.Forall (x, g) -> NNF.Forall (x, nnf g)
    | PureFormula.Exists (x, g) -> NNF.Exists (x, nnf g)

and neg (f: PureFormula) : NNF =
    match f with
    | PureFormula.Verum         -> NNF.Falsum
    | PureFormula.Falsum        -> NNF.Verum
    | PureFormula.Atom (p, ts)  -> NNF.Lit (Neg (p, ts))
    | PureFormula.Not g         -> nnf g
    | PureFormula.And (g, h)    -> NNF.Or  (neg g, neg h)
    | PureFormula.Or  (g, h)    -> NNF.And (neg g, neg h)
    | PureFormula.Forall (x, g) -> NNF.Exists (x, neg g)
    | PureFormula.Exists (x, g) -> NNF.Forall (x, neg g)

In [ ]:
f2

In [ ]:
let f3 = nnf f2
formulaToString (nnfToFormula f3)

## Renaming Bound Variables

The function `termVariables` takes one argument:
- `t` is a term.

It returns the set of all variables occurring in `t`.  The function `allVariables` takes a formula `f` in negation normal form and returns the set of all variables occurring in `f`.  This set contains both the variables occurring in terms and the variables bound by quantifiers.

In [ ]:
let rec termVariables (t: Term) : Set<string> =
    match t with
    | Var x       -> set [ x ]
    | Fun (_, ts) -> Set.unionMany (List.map termVariables ts)

let rec allVariables (f: NNF) : Set<string> =
    match f with
    | NNF.Lit (Pos (_, ts)) | NNF.Lit (Neg (_, ts)) -> Set.unionMany (List.map termVariables ts)
    | NNF.Verum | NNF.Falsum                        -> Set.empty
    | NNF.And (g, h) | NNF.Or (g, h)                -> allVariables g + allVariables h
    | NNF.Forall (x, g) | NNF.Exists (x, g)         -> (allVariables g).Add x

In [ ]:
allVariables f3

The function `applyLiteral` takes two arguments:
- `σ` is a substitution,
- `l` is a literal.

It applies `σ` to all terms occurring in `l`.

In [ ]:
let applyLiteral (σ: Map<string, Term>) (l: Literal) : Literal =
    match l with
    | Pos (p, ts) -> Pos (p, List.map (applyTerm σ) ts)
    | Neg (p, ts) -> Neg (p, List.map (applyTerm σ) ts)

The function `rectify` takes one argument:
- `f` is a formula in negation normal form.

It returns a *rectified* formula that is equivalent to `f`.  To this end, `f` is traversed from left to right.  If a quantifier binds a variable `x` that has already been bound by a quantifier encountered earlier, then `x` is replaced by a **new** variable, both in the quantifier and in its scope.  All other quantifiers keep their variables.  The new variables are single lower case letters that do not occur in `f`.  This only works if there are enough letters that do not already occur in $f$.  This approach would not be good enough for a production quality program,
but for the case of a demonstration it is sufficient.  The alternative would be to rename the variables as `x1`, `x2`, `x3`, $\cdots$, but that becomes unreadable very fast.

**Implementation:**
- `Vars` is the set of all variables occurring in `f`.  As the new variables do not occur in `Vars`, they can not clash with variables that are bound later in `f` or that occur free in `f`.
- The local function `newVariable` takes a variable `x` and the set `Bound` of the variables that have been bound so far.  If `x` has not been bound yet, `x` is kept.  Otherwise, the first letter that occurs neither in `Vars` nor in `Bound` is returned.
- The local function `rename` takes a substitution `σ`, the set `Bound`, and a formula `g` that is a part of `f`.  The substitution `σ` maps every variable that is bound by a quantifier surrounding `g` to its (possibly new) name.  `rename` returns a pair: the first component is the renamed formula, the second component is the set of variables that have been bound after `g` has been processed.  This set is passed from the left subformula to the right subformula of a conjunction or disjunction, so that a variable bound on the left is renamed when it is bound again on the right.

In [ ]:
let rectify (f: NNF) : NNF =
    let Vars = allVariables f
    let newVariable (x: string) (Bound: Set<string>) : string =
        if not (Bound.Contains x) then
            x
        else
            List.find (fun v -> not (Vars.Contains v || Bound.Contains v))
                      [ for c in 'a' .. 'z' -> string c ]
    let rec rename (σ: Map<string, Term>) (Bound: Set<string>) (g: NNF) : NNF * Set<string> =
        match g with
        | NNF.Lit l              -> NNF.Lit (applyLiteral σ l), Bound
        | NNF.Verum | NNF.Falsum -> g, Bound
        | NNF.And (g1, g2) ->
            let r1, Bound1 = rename σ Bound  g1
            let r2, Bound2 = rename σ Bound1 g2
            NNF.And (r1, r2), Bound2
        | NNF.Or (g1, g2) ->
            let r1, Bound1 = rename σ Bound  g1
            let r2, Bound2 = rename σ Bound1 g2
            NNF.Or (r1, r2), Bound2
        | NNF.Forall (x, h) ->
            let y = newVariable x Bound
            let r, Bound1 = rename (σ.Add (x, Var y)) (Bound.Add y) h
            NNF.Forall (y, r), Bound1
        | NNF.Exists (x, h) ->
            let y = newVariable x Bound
            let r, Bound1 = rename (σ.Add (x, Var y)) (Bound.Add y) h
            NNF.Exists (y, r), Bound1
    fst (rename Map.empty Set.empty f)

In [ ]:
let f4 = rectify f3
formulaToString (nnfToFormula f4)

The example formula from the beginning of this notebook is rectified as follows:

In [ ]:
formulaToString (nnfToFormula (rectify (nnf (purify (parse "(∀x:P(x)) ∨ (∀x:Q(x))")))))

## Prenex Normal Form

The functions given below only work if all quantifiers that occur in a formula bind **different** variables, i.e. if the formulas are *rectified*.  If this assumption is not satisfied, then they will produce <u>garbage</u>.  This is the reason why we have renamed the bound variables in the previous section.

A *quantifier prefix* is a list of quantifiers together with the variables they bind, i.e. a list of the form
$$ [Q_1\,x_1; \cdots; Q_n\,x_n] $$
Here, the $Q_i$ denote quantifiers, i.e. we have $Q_i \in \{\forall, \exists\}$, while the $x_i$ are variables.  We represent an element $Q_i\,x_i$ of a quantifier prefix by a value of the type `Quantifier`.

In [ ]:
type Quantifier =
    | All of string   // ∀x
    | Ex  of string   // ∃x

The *matrix* of a formula in prenex normal form is the part that remains when the quantifier prefix is removed.  Hence, the matrix is a formula in negation normal form that does not contain any quantifiers.  The type `Matrix` represents these formulas.  Its constructors are the constructors of the type `NNF` except for `Forall` and `Exists`.

A formula in prenex normal form is represented as a pair `(Qs, m)`, where `Qs` is a quantifier prefix and `m` is a matrix.

In [ ]:
[<RequireQualifiedAccess>]
type Matrix =
    | Lit    of Literal            // P(t1, ..., tn) or ¬P(t1, ..., tn)
    | Verum                        // ⊤
    | Falsum                       // ⊥
    | And    of Matrix * Matrix    // ∧
    | Or     of Matrix * Matrix    // ∨

The function `matrixToFormula` takes one argument:
- `m` is a matrix.

It returns `m` as a value of the type `Formula`.  We need this function in order to display matrices.

In [ ]:
let rec matrixToFormula (m: Matrix) : Formula =
    match m with
    | Matrix.Lit l      -> literalToFormula l
    | Matrix.Verum      -> Verum
    | Matrix.Falsum     -> Falsum
    | Matrix.And (g, h) -> And (matrixToFormula g, matrixToFormula h)
    | Matrix.Or  (g, h) -> Or  (matrixToFormula g, matrixToFormula h)

The function `mergeQuantifiers` takes two arguments:
- `Q1` is a quantifier prefix,
- `Q2` is a quantifier prefix.

It merges them into a new quantifier prefix such that the relative order of the quantifiers remains the same, i.e. if both $Q_1\,x_1$ and $Q_2\,x_2$ occur in `Q1` and $Q_1\,x_1$ occurs before $Q_2\,x_2$, then $Q_1\,x_1$ will occur before $Q_2\,x_2$ in the result.  Existential quantifiers are extracted first, since this results in Skolem functions with fewer arguments.

In [ ]:
let rec mergeQuantifiers (Q1: Quantifier list) (Q2: Quantifier list) : Quantifier list =
    match Q1, Q2 with
    | [], _              -> Q2
    | _, []              -> Q1
    | (Ex x) :: rest, _  -> Ex x :: mergeQuantifiers rest Q2   // extract existential quantifiers first
    | _, (Ex y) :: rest  -> Ex y :: mergeQuantifiers Q1 rest
    | q :: rest, _       -> q :: mergeQuantifiers rest Q2

In [ ]:
mergeQuantifiers [ All "x"; Ex "y" ] [ Ex "u"; All "v" ]

The function `extractQuantifiers` takes one argument:
- `f` is a rectified formula in negation normal form.

It returns a pair `(Qs, m)`, where `Qs` is a quantifier prefix and `m` is the <em style="color:blue;">matrix</em> of the formula $f$.  The matrix of a formula is the part that remains when all quantifiers have been extracted.  Hence, the pair `(Qs, m)` represents the *prenex normal form* of `f`.

In [ ]:
let rec extractQuantifiers (f: NNF) : Quantifier list * Matrix =
    match f with
    | NNF.Lit l      -> [], Matrix.Lit l
    | NNF.Verum      -> [], Matrix.Verum
    | NNF.Falsum     -> [], Matrix.Falsum
    | NNF.And (g, h) ->
        let qg, gm = extractQuantifiers g
        let qh, hm = extractQuantifiers h
        // this works because f is rectified
        mergeQuantifiers qg qh, Matrix.And (gm, hm)
    | NNF.Or (g, h) ->
        let qg, gm = extractQuantifiers g
        let qh, hm = extractQuantifiers h
        mergeQuantifiers qg qh, Matrix.Or (gm, hm)
    | NNF.Forall (x, g) ->
        let qg, gm = extractQuantifiers g
        All x :: qg, gm
    | NNF.Exists (x, g) ->
        let qg, gm = extractQuantifiers g
        Ex x :: qg, gm

In [ ]:
f4

In [ ]:
let qs, f5 = extractQuantifiers f4
qs, formulaToString (matrixToFormula f5)

The function `attachQuantifiers` takes two arguments:
- `Qs` is a quantifier prefix,
- `m` is a matrix.

It combines the quantifiers `Qs` and the matrix `m` into a quantified formula of type `Formula`.  Since we represent the prenex normal form as a pair, we need this function only in order to display the prenex normal form.

In [ ]:
let rec attachQuantifiers (Qs: Quantifier list) (m: Matrix) : Formula =
    match Qs with
    | []            -> matrixToFormula m
    | All x :: rest -> Forall (x, attachQuantifiers rest m)
    | Ex  x :: rest -> Exists (x, attachQuantifiers rest m)

In [ ]:
formulaToString (attachQuantifiers qs f5)

## Skolemization (Eliminating Existential Quantifiers)

The mutable variable `skolemCounter` is a global variable that is needed to create unique Skolem functions.  The function `skolemFunction` takes no arguments and returns a new name for a Skolem function every time it is called.

In [ ]:
let mutable skolemCounter = 0

let skolemFunction () : string =
    skolemCounter <- skolemCounter + 1
    "sk" + string skolemCounter

The function `applyMatrix` takes two arguments:
- `σ` is a substitution,
- `m` is a matrix.

It applies `σ` to all terms occurring in `m`.  As a matrix does not contain quantifiers, there are no bound variables that would have to be taken care of.

In [ ]:
let rec applyMatrix (σ: Map<string, Term>) (m: Matrix) : Matrix =
    match m with
    | Matrix.Lit l                 -> Matrix.Lit (applyLiteral σ l)
    | Matrix.Verum | Matrix.Falsum -> m
    | Matrix.And (g, h)            -> Matrix.And (applyMatrix σ g, applyMatrix σ h)
    | Matrix.Or  (g, h)            -> Matrix.Or  (applyMatrix σ g, applyMatrix σ h)

The function `skolemize` takes three arguments:
- `Qs` is a quantifier prefix,
- `m` is a matrix, i.e. the pair `(Qs, m)` is a formula in <em style="color:blue">prenex normal form</em>,
- `Vs` is a list of variables that are universally quantified.

It <em style="color:blue">skolemizes</em> the formula `(Qs, m)`, i.e. it replaces all existentially quantified variables by appropriate <em style="color:blue">Skolem functions</em>.  The arguments of these Skolem functions are the variables from `Vs`.  When `skolemize` is called initially, `Vs` is the empty list.  During the recursion, `Vs` collects the universally quantified variables that precede the current quantifier.  The result is a matrix: the universal quantifiers are dropped, since the variables occurring in clauses are implicitly universally quantified.

For skolemization to work correctly, we have to assume that
<font size="4" style="color:darkgreen; size:125%">the formula does not contain free variables</font>!

In [ ]:
let rec skolemize (Qs: Quantifier list) (m: Matrix) (Vs: string list) : Matrix =
    match Qs with
    | [] -> m
    | Ex x :: rest ->
        let t = Fun (skolemFunction (), List.map Var Vs)
        skolemize rest (applyMatrix (Map [ (x, t) ]) m) Vs
    | All x :: rest ->
        skolemize rest m (Vs @ [ x ])

In [ ]:
qs, f5

In [ ]:
let f6 = skolemize qs f5 []
formulaToString (matrixToFormula f6)

## Conversion to Clauses

The function `cnf` takes one argument:
- `m` is a matrix that results from <em style="color:blue">skolemization</em>, i.e. a formula in <em style="color:blue">negation normal form</em> that does not contain quantifiers.

It returns the <em style="color:blue">conjunctive normal form</em> of `m` in <em style="color:blue">set notation</em>.  This works the same way as in propositional logic.  Since every value of the type `Matrix` is in negation normal form and does not contain quantifiers, the pattern match covers all possible cases and `cnf` can not fail.

In [ ]:
let rec cnf (m: Matrix) : Set<Clause> =
    match m with
    | Matrix.Lit l      -> set [ set [ l ] ]
    | Matrix.Verum      -> Set.empty
    | Matrix.Falsum     -> set [ Set.empty ]
    | Matrix.And (g, h) -> cnf g + cnf h
    | Matrix.Or  (g, h) -> set [ for k1 in cnf g do for k2 in cnf h -> k1 + k2 ]

In [ ]:
f6

In [ ]:
let f7 = cnf f6
printfn "%s" (clausesToString f7)

## Putting Everything Together

The function `normalize` takes one argument:
- `f` is a formula from first order logic that **must not** contain free variables.

It transforms $f$ into a set of first order clauses that is satisfiable if and only if `f` is satisfiable.

The types of the functions ensure that the steps are performed in the right order: `purify` maps a `Formula` to a `PureFormula`, `nnf` maps a `PureFormula` to an `NNF`, `rectify` maps an `NNF` to a rectified `NNF`, `extractQuantifiers` maps an `NNF` to a quantifier prefix and a `Matrix`, `skolemize` returns a `Matrix`, and `cnf` maps a `Matrix` to a set of clauses.

In [ ]:
let normalize (f: Formula) : Set<Clause> =
    let f1     = purify f
    let f2     = nnf f1
    let f3     = rectify f2
    let qs, f4 = extractQuantifiers f3
    let f5     = skolemize qs f4 []
    cnf f5

In [ ]:
normalize f1

The function `test` takes one argument:
- `s` is a string representing a formula without free variables.

It computes the clauses of this formula and prints them.

In [ ]:
let test (s: string) : unit =
    let f = parse s
    printfn "The CNF of %s is:" s
    printfn "%s" (clausesToString (normalize f))

In [ ]:
test s

In [ ]:
test "¬(∃y:∀x:P(x,y)→∀u:∃v:P(u,v))"

The following formula is satisfiable: in a structure with two elements, where the predicate $P$ holds for exactly one of them, both conjuncts are true.  Hence, its set of clauses has to be satisfiable, too.  When `↔` is eliminated, the formula $\forall x: P(x)$ is duplicated.  If the variable $x$ in the second copy were renamed to a variable that already occurs elsewhere in the formula, e.g. to $a$, then the prenex normal form would identify this variable with the variable $a$ that is bound by $\exists a$.  We would then get the unsatisfiable set of clauses $\bigl\{\{P(c)\}, \{\neg P(c)\}\bigr\}$ for a single Skolem constant $c$.  This is why `rectify` only uses variables that do not occur in the formula.

In [ ]:
test "(∃a:P(a)) ∧ (⊥ ↔ ∀x:P(x))"